# Transaction Foundation Model on Ray — Part 5: Batch embedding extraction

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: ~15 min on the GPU workers)

---

Previously in Part 4, we trained the foundation model. With the foundation model built, now build the first fraud detector. In this first approach, we will use **embeddings** — the model's understanding of a transaction, written as a list of numbers. This notebook produces the embeddings, and Part 6 trains on them and evaluates the results against the baselines from NVIDIA.

Later, Part 7 builds a stronger detector by fine-tuning the foundation model itself.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import logging

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},
         logging_level=logging.ERROR)

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; full = every card
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Turn the trained model into one vector per transaction

The foundation model pretrained on long histories, but Part 6's fraud classifiers need a fixed-length vector per transaction. Following NVIDIA's blueprint, we embed each transaction on its own: feed `<bos>` + its 12 field tokens + `<eos>` through the model, and the model's internal state at the last token is the embedding. Embedding a transaction alone means the vector is a learned re-encoding of the same 13 raw fields — what the model adds is what it learned about transactions in general during pretraining, not context from this card's history. Part 7 revisits that choice with history windows.

We embed three sets. Train is balance-sampled: every fraud, plus normals up to a cap — 1M rows at `full`, NVIDIA's recipe. The fraud detectors see fraud often enough to learn from it, and we skip embedding most of the 19.5M train transactions. Val and test are the 100K stratified eval subsets from Part 2, embedded whole. We write each split as `embed_<split>.npy` + `lbl_<split>.npy` + `raw_<split>.parquet` (the 13 native fields), so Part 6 trains its detectors without touching the tokenizer again.

## Stream CPU work into GPU forward passes

This stage needs two kinds of hardware at once: tokenizing transactions is CPU work and the forward pass is GPU work. The model is also expensive to load, so it should load once per GPU and stay in memory. The pipeline gives each piece its own resources:

- One **CPU task** per split draws the seeded balanced sample, sorts it into final row order, and writes `lbl_`/`raw_` immediately — labels and features are pure CPU work.
- **CPU workers** stream batches of transactions into token rows (`encode_txn_batch`, the same tokenizer as Part 3).
- **GPU actors** do only forward passes. Passing a class (`GPUEmbedder`) to `map_batches` is what creates them: Ray Data builds an actor pool, each actor's constructor runs once and loads the model, and batches then stream through — the standard Ray Data pattern for GPU inference.

The pools scale independently: more CPU workers keep the GPUs busy, more GPU actors raise throughput linearly, and neither holds hardware the other needs. Each row carries its output position, so the results come out in the same order as the single-GPU reference, and the output is verified against it (the checks are in `scripts/`).

### Run the embedding pipeline

One loop runs the three splits.

In [2]:
from src.nvembed import (prepare_embed_split, encode_txn_batch, GPUEmbedder,
                         assemble_embeddings)

eb = cfg["embed"]
use_gpu = eb["use_gpu"]                    # mini: CPU actors; full: one actor per A10G
if not os.path.exists(os.path.join(paths["embeddings"], "embed_test.npy")):
    for split, fname in {"train": None, "val": "val_eval.parquet",
                         "test": "test_eval.parquet"}.items():
        # One Ray task per split: draw the seeded balanced sample, sort it into
        # final row order, and write lbl_<split>.npy + raw_<split>.parquet.
        prep = ray.get(prepare_embed_split.remote(
            paths["nvsplit"], fname, paths["embeddings"], split,
            eb["balanced_train"], 42))

        # The streaming pipeline. CPU workers tokenize each batch, GPU actors run
        # the forward pass, and the results land as parquet shards.
        shards = os.path.join(paths["embeddings"], f"_emb_{split}")
        ray.data.read_parquet(prep["prep"]) \
            .map_batches(encode_txn_batch, batch_format="pandas") \
            .map_batches(GPUEmbedder,
                         fn_constructor_kwargs={"hf_dir": paths["hf"],
                                                "batch_size": eb["batch_size"]},
                         batch_format="numpy",
                         batch_size=4096,   # rows per actor call; GPU microbatch = eb["batch_size"]
                         num_gpus=(1 if use_gpu else 0),
                         concurrency=eb["num_workers"]) \
            .write_parquet(shards)

        # assemble_embeddings runs one Ray task that gathers the shards into
        # embed_<split>.npy in row order, then removes the temp files. (12 lines in src.)
        meta = assemble_embeddings(shards, os.path.join(paths["embeddings"],
                                                        f"embed_{split}.npy"),
                                   prep["prep"], embed_dim=cfg["model"]["d_model"])
        print(split, {**{k: prep[k] for k in ("rows", "fraud")}, **meta})
else:
    print(f"embeddings present at {paths['embeddings']} — skipping (delete the dir to re-embed)")

embeddings present at /mnt/cluster_storage/transaction-fm/embeddings/mini/ — skipping (delete the dir to re-embed)


## Check the embeddings for collapse

The classic self-supervised failure is silent **representation collapse**: every input maps to nearly the same vector while the training loss looks fine. Two cheap numbers guard against it — mean pairwise cosine similarity (1.0 means collapse) and mean per-feature variance (0 means collapse).

Transformer embeddings normally show high mean cosine — the vectors cluster in direction — while still separating well in the dimensions that matter. A high mean cosine with healthy variance is typical, not near-collapse; the full-scale embeddings are near 0.9. This check catches the degenerate case; the real test of embedding quality is the fraud comparison in Part 6.

In [3]:
# Representation-collapse check on the test embeddings: mean pairwise cosine similarity
# (→1.0 = collapse) and mean feature variance (→0 = collapse).
emb = np.load(os.path.join(paths["embeddings"], "embed_test.npy"))
lbl = np.load(os.path.join(paths["embeddings"], "lbl_test.npy"))

rng = np.random.RandomState(0)
idx = rng.choice(len(emb), size=min(2000, len(emb)), replace=False)
s = emb[idx].astype(np.float32)
s /= (np.linalg.norm(s, axis=1, keepdims=True) + 1e-8)
cos = s @ s.T
mean_cos = float((cos.sum() - len(idx)) / (len(idx) * (len(idx) - 1)))

print(f"{len(emb):,} test embeddings · dim {emb.shape[1]}  ({int(lbl.sum()):,} fraud)")
print(f"mean pairwise cosine  {mean_cos:.3f}   (→1.0 = collapse)")
print(f"mean feature variance {float(emb.var(0).mean()):.4f}   (→0 = collapse)")
print(f"example embedding[:8] = {[round(float(x), 3) for x in emb[0, :8]]}  (label {int(lbl[0])})")

100,000 test embeddings · dim 64  (108 fraud)
mean pairwise cosine  0.729   (→1.0 = collapse)
mean feature variance 0.2654   (→0 = collapse)
example embedding[:8] = [-1.843, 0.358, 0.152, -1.225, -0.139, -0.797, -0.474, -0.513]  (label 0)


## Scaling factors

Embedding cost is linear in the transaction count and the model size. The work is forward passes only — no gradients, no optimizer state — so GPU memory goes to batch size instead of training bookkeeping. At `full` the stage embeds ~1.2M transactions (the 1M balanced train sample plus the two 100K eval sets) in about fifteen minutes on the GPU pool.

The two pools tune against each other. Tokenization produces rows at a rate set by the CPU worker count; each GPU actor consumes them at a rate set by the model and batch size. If the GPUs idle, add CPU workers; if tokenized batches queue, add GPU actors. The two counts move independently in the config.

This stage's cost also recurs: pretraining happens occasionally, but re-embedding runs every time fresh transactions arrive. In production, this throughput sets the recurring bill.

## Takeaways

We embedded every split and wrote the results to shared storage — `embed_`, `lbl_`, and `raw_` files per split. Part 6 trains fraud detectors on these files directly.

One streaming pipeline did the work. CPU workers tokenized, GPU actors ran the forward passes, and the two pools scale independently. The outputs are verified against the single-GPU reference, and the notebook composes the same functions the headless job runs, so the two cannot drift.

---

## Next

**Part 6 — Fraud detectors**: train XGBoost on the raw fields, on these embeddings, and on both together — and compare against NVIDIA's published results.